##                            Employee Performance Score Prediction using PyTorch

In [1]:
import pandas as pd

In [2]:
#loading csv file
df = pd.read_csv("employee_performance_dataset.csv")
df.head()

,Experience_Years,Training_Hours_Monthly,Projects_Completed,Attendance_Percent,Performance_Score
0,6,12,9,80.67,94.07
1,19,32,18,80.90,100.00
2,14,4,23,84.01,100.00
3,10,36,20,70.73,100.00
4,7,13,15,75.33,100.00


In [3]:
df.shape

(10000, 5)

##### Train-Test Split

In [4]:
from sklearn.model_selection import train_test_split

X = df.drop('Performance_Score', axis = 1).values
y = df['Performance_Score'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

##### Training a neural network

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [7]:
device = torch.device("cpu")
print(device)

cpu


In [8]:
X_train_tensor.shape

torch.Size([8000, 4])

In [9]:
y_train_tensor = y_train_tensor.reshape(-1, 1)
y_test_tensor = y_test_tensor.reshape(-1, 1)

In [10]:
class PerformancePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )
    def forward(self, x):
        return self.network(x)

In [11]:
torch.manual_seed(42)
model = PerformancePredictor()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [12]:
with torch.no_grad():
    pred = model(X_train_tensor)
    print("Initial loss:", criterion(pred, y_train_tensor).item())

Initial loss: 9038.142578125


In [13]:
# Training Loop
epochs = 2000

for epoch in range(epochs):
    # forward pass: Compute Performance Score
    predictions = model(X_train_tensor)
    loss = criterion(predictions, y_train_tensor)

    # backward pass: Compute gradients
    optimizer.zero_grad()
    loss.backward()

    # update weights
    optimizer.step()

    if(epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.2f}")

Epoch [100/2000], Loss: 8903.75
Epoch [200/2000], Loss: 8308.29
Epoch [300/2000], Loss: 6759.67
Epoch [400/2000], Loss: 4340.54
Epoch [500/2000], Loss: 2058.15
Epoch [600/2000], Loss: 820.20
Epoch [700/2000], Loss: 419.73
Epoch [800/2000], Loss: 316.94
Epoch [900/2000], Loss: 283.00
Epoch [1000/2000], Loss: 262.36
Epoch [1100/2000], Loss: 244.37
Epoch [1200/2000], Loss: 227.12
Epoch [1300/2000], Loss: 210.32
Epoch [1400/2000], Loss: 193.62
Epoch [1500/2000], Loss: 176.51
Epoch [1600/2000], Loss: 157.19
Epoch [1700/2000], Loss: 135.70
Epoch [1800/2000], Loss: 114.25
Epoch [1900/2000], Loss: 94.28
Epoch [2000/2000], Loss: 76.55


In [14]:
# model evaluation
model.eval()

with torch.no_grad():
    test_pred = model(X_test_tensor)
    test_loss = criterion(test_pred, y_test_tensor)

print(f"Test Loss: {test_loss.item():.4f}")

Test Loss: 66.5675


### Using GPU as Device

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [16]:
torch.manual_seed(42)
model_gpu = PerformancePredictor().to(device)

In [17]:
criterion_gpu = nn.MSELoss()

optimizer_gpu = optim.Adam(
    model_gpu.parameters(),
    lr=0.0015
)

In [18]:
X_train_tensor_gpu = X_train_tensor.to(device)
y_train_tensor_gpu = y_train_tensor.to(device)

X_test_tensor_gpu = X_test_tensor.to(device)
y_test_tensor_gpu = y_test_tensor.to(device)

In [19]:
y_train_tensor_gpu = y_train_tensor_gpu.reshape(-1, 1)
y_test_tensor_gpu = y_test_tensor_gpu.reshape(-1, 1)

In [20]:
print(X_train_tensor_gpu.device)

cuda:0


In [21]:
with torch.no_grad():
    pred = model_gpu(X_train_tensor_gpu)
    print("Initial loss:", criterion_gpu(pred, y_train_tensor_gpu).item())

Initial loss: 9038.1435546875


In [22]:
epochs = 2000

for epoch in range(epochs):

    model_gpu.train()

    predictions = model_gpu(X_train_tensor_gpu)

    loss = criterion_gpu(predictions, y_train_tensor_gpu)

    optimizer_gpu.zero_grad()
    loss.backward()
    optimizer_gpu.step()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

Epoch 100: Loss = 8745.4209
Epoch 200: Loss = 7030.2754
Epoch 300: Loss = 3393.3433
Epoch 400: Loss = 894.2386
Epoch 500: Loss = 364.4890
Epoch 600: Loss = 284.8604
Epoch 700: Loss = 255.8228
Epoch 800: Loss = 233.2373
Epoch 900: Loss = 209.9089
Epoch 1000: Loss = 186.3243
Epoch 1100: Loss = 162.0512
Epoch 1200: Loss = 133.9603
Epoch 1300: Loss = 105.0511
Epoch 1400: Loss = 79.7629
Epoch 1500: Loss = 59.5467
Epoch 1600: Loss = 43.5188
Epoch 1700: Loss = 31.4468
Epoch 1800: Loss = 23.1881
Epoch 1900: Loss = 18.3533
Epoch 2000: Loss = 15.5168


In [23]:
from sklearn.metrics import r2_score, mean_squared_error

model.eval()

with torch.no_grad():
    pred = model(X_test_tensor)


pred = pred.numpy().flatten()
true = y_test_tensor.numpy().flatten()

r2 = r2_score(true, pred)
rmse = mean_squared_error(true, pred) ** 0.5

print(f"R² Score : {r2:.4f}")
print(f"RMSE     : {rmse:.4f}")

R² Score : 0.5148
RMSE     : 8.1589


In [24]:
model_gpu.eval()

with torch.no_grad():
    pred = model_gpu(X_test_tensor_gpu)

pred = pred.cpu().numpy()
true = y_test_tensor_gpu.cpu().numpy().flatten()

print("R²:", r2_score(true, pred))
print("RMSE:", mean_squared_error(true, pred) ** 0.5)

R²: 0.889336347579956
RMSE: 3.89660882966188
